In [7]:
import json
from building_with_claude_api import MessageHelper, PromptEvaluator

mh = MessageHelper()
pe = PromptEvaluator()

#### Generating Test Data with Code

Now we'll create our dataset generation function:

In [ ]:
def generate_dataset():
  prompt = """
Generate an evaluation dataset for a prompt that writes a compact, concise
1-day meal plan for a single athlete. Generate an array of JSON objects, each
representing one task. Every task must state these athlete details:

* height (in cm)
* weight (in kg)
* goal (e.g. muscle gain, fat loss, endurance)
* dietary restrictions (or none)

Example output:
```json
[
  {
    "task": "Create a 1-day meal plan for a 180cm, 75kg athlete aiming for muscle gain with no dietary restrictions"
  },
  ...additional
]
```

Please generate 3 objects.
"""

  return prompt

To properly parse the JSON response, we'll use prefilling and stop sequences:

In [9]:
messages = []
prompt = generate_dataset()
mh.add_user_message(messages, prompt)
mh.add_assistant_message(messages, "```json")
text = mh.chat(messages, stop_sequences=["```"])

dataset = json.loads(text)

In [10]:
with open('../datasets/dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

### Running the eval

##### Running the Evaluation

To execute our evaluation pipeline, we load our dataset and run it through our functions:

In [ ]:
criteria = """The output should include:
- Daily caloric total
- Macronutrient breakdown
- Meals with exact foods, portions, and timing"""

with open("../datasets/dataset.json", "r") as f:
    dataset = json.load(f)

results = pe.run_eval(dataset, criteria)

In [12]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# 1-Day Muscle Gain Meal Plan\n**Athlete Profile:** 180cm, 75kg | Goal: Muscle Gain | Caloric Target: ~3,000-3,200 kcal | Protein: ~150g\n\n---\n\n## **BREAKFAST** (7:00 AM)\n- 3 whole eggs + 2 egg whites, scrambled\n- 80g oatmeal (dry weight) with berries\n- 1 banana\n- 1 tbsp almond butter\n\n**Macros:** 650 kcal | 28g protein | 85g carbs | 18g fat\n\n---\n\n## **MID-MORNING SNACK** (10:00 AM)\n- Greek yogurt (200g, 0% fat)\n- 40g granola\n- 30g mixed nuts\n\n**Macros:** 420 kcal | 20g protein | 38g carbs | 18g fat\n\n---\n\n## **LUNCH** (1:00 PM)\n- 200g grilled chicken breast\n- 250g white rice (cooked)\n- 150g broccoli with 1 tbsp olive oil\n- Seasoning (salt, pepper, garlic)\n\n**Macros:** 750 kcal | 52g protein | 85g carbs | 12g fat\n\n---\n\n## **PRE-WORKOUT SNACK** (3:30 PM)\n*(30-45 min before training)*\n- 2 slices whole wheat bread\n- 30g peanut butter\n- 1 apple\n\n**Macros:** 420 kcal | 16g protein | 48g carbs | 16g fat\n\n---\n\n## **POST-WORKOUT** (